# YOLO26 베이스라인

목표: 데이터 변환 → 학습 → 제출 csv까지 한 번 완주.
분할은 EDA 때 만든 것(시드 410) 그대로 씀.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import torch
import ultralytics

# 레포 루트 찾기 (pyproject.toml 있는 폴더)
ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent

# 압축 푼 방식이 달라도 되게 train_images 있는 폴더를 데이터 폴더로
DATA_DIR = sorted((ROOT / "data" / "raw").rglob("train_images"))[0].parent

# GPU 있으면 0번 GPU, 없으면 CPU
DEVICE = 0 if torch.cuda.is_available() else "cpu"

print("ultralytics 버전:", ultralytics.__version__)
print("device:", DEVICE)
print("DATA_DIR:", DATA_DIR)

## 1. 라벨 읽기, 클래스 번호표

YOLO는 클래스를 0부터 순서대로 씀. 원래 약 번호(1900 등)를 작은 순으로 0, 1, 2…에 대응시킨다.
제출할 때는 같은 표로 되돌림.

In [ ]:
TRAIN_IMG_DIR = DATA_DIR / "train_images"
TRAIN_ANN_DIR = DATA_DIR / "train_annotations"
TEST_IMG_DIR = DATA_DIR / "test_images"

rows = []
cat_names = {}  # 약 번호 → 약 이름
for jp in sorted(TRAIN_ANN_DIR.rglob("*.json")):
    with open(jp, encoding="utf-8") as f:
        d = json.load(f)
    img = d["images"][0]
    for c in d["categories"]:
        cat_names[c["id"]] = c["name"]
    for a in d["annotations"]:
        rows.append({
            "file_name": img["file_name"],
            "img_w": img["width"],
            "img_h": img["height"],
            "category_id": a["category_id"],
            "bbox": a["bbox"],  # [x, y, w, h] 픽셀, 왼쪽 위 기준
        })
ann = pd.DataFrame(rows)

# 번호표: 원래 약 번호(작은 순) → 0, 1, 2 ...
CAT_IDS = sorted(cat_names)
CAT2YOLO = {c: i for i, c in enumerate(CAT_IDS)}
YOLO2CAT = {i: c for c, i in CAT2YOLO.items()}

print("박스:", len(ann), "/ 이미지:", ann["file_name"].nunique(), "/ 약 종류:", len(CAT_IDS))
print("번호표 앞부분:", list(CAT2YOLO.items())[:5])

## 2. train/val 분할

EDA 때 만든 분할(시드 410) 불러옴. 조합 단위, 조합 1개뿐인 약은 train 고정.

In [ ]:
split = pd.read_csv(ROOT / "data" / "processed" / "split.csv")
print(split["split"].value_counts())

# 박스마다 train/val 붙이기
ann = ann.merge(split[["file_name", "split"]], on="file_name", how="left")
print()
print("분할 정보 없는 박스:", ann["split"].isna().sum())  # 0이어야 정상

## 3. YOLO 형식으로 변환

txt 한 줄 = 박스 하나: 클래스번호 중심x 중심y 너비 높이 (0~1 비율)
이미지 밖으로 나간 박스는 경계에 맞춰 자르고, 완전히 밖이면 버림.
클래스 이름은 한글 깨짐 방지로 약 번호를 씀.

In [ ]:
import shutil
import yaml

YOLO_DIR = ROOT / "data" / "processed" / "yolo"

# 다시 돌릴 때 예전 파일이 남지 않게 지우고 새로 만듦
if YOLO_DIR.exists():
    shutil.rmtree(YOLO_DIR)
for s in ["train", "val"]:
    (YOLO_DIR / "images" / s).mkdir(parents=True)
    (YOLO_DIR / "labels" / s).mkdir(parents=True)

dropped = 0
for name, g in ann.groupby("file_name"):
    s = g["split"].iloc[0]
    shutil.copy(TRAIN_IMG_DIR / name, YOLO_DIR / "images" / s / name)

    lines = []
    for _, r in g.iterrows():
        x, y, w, h = r["bbox"]
        W, H = r["img_w"], r["img_h"]
        # 이미지 밖으로 나간 부분 잘라내기
        x1, y1 = max(0, x), max(0, y)
        x2, y2 = min(W, x + w), min(H, y + h)
        if x2 <= x1 or y2 <= y1:
            dropped += 1  # 완전히 밖이면 버림
            continue
        cx, cy = (x1 + x2) / 2 / W, (y1 + y2) / 2 / H
        bw, bh = (x2 - x1) / W, (y2 - y1) / H
        lines.append(f"{CAT2YOLO[r['category_id']]} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}")

    (YOLO_DIR / "labels" / s / (Path(name).stem + ".txt")).write_text("\n".join(lines))

# 설정 파일
data_cfg = {
    "path": str(YOLO_DIR),
    "train": "images/train",
    "val": "images/val",
    "names": {i: str(c) for i, c in YOLO2CAT.items()},
}
with open(YOLO_DIR / "data.yaml", "w", encoding="utf-8") as f:
    yaml.safe_dump(data_cfg, f, sort_keys=False)

for s in ["train", "val"]:
    print(s, "이미지:", len(list((YOLO_DIR / "images" / s).glob("*.png"))),
          "/ 라벨:", len(list((YOLO_DIR / "labels" / s).glob("*.txt"))))
print("버린 박스:", dropped)

재원님 라벨 수정 목록 적용 (jaeone_label_fixes.csv)
- add: 빠진 박스 추가
- bbox: 좌표 수정
- area: 넓이 값 오타라 학습엔 영향 없음, 건너뜀

In [ ]:
fix_path = ROOT / "notebooks" / "eda" / "label_fix" / "jaeone_label_fixes.csv"
fixes = pd.read_csv(fix_path, encoding="utf-8-sig")  # 파일 앞에 BOM이 있어서 utf-8-sig로 읽음

img_size = ann.drop_duplicates("file_name").set_index("file_name")[["img_w", "img_h"]]

applied = {"add": 0, "bbox": 0, "area": 0}
for _, f in fixes.iterrows():
    if f["action"] == "add":
        w, h = img_size.loc[f["file_name"]]
        new_row = {"file_name": f["file_name"], "img_w": w, "img_h": h,
                   "category_id": int(f["category_id"]), "bbox": json.loads(f["bbox_after"])}
        ann = pd.concat([ann, pd.DataFrame([new_row])], ignore_index=True)
    elif f["action"] == "bbox":
        hit = (ann["file_name"] == f["file_name"]) & (ann["category_id"] == f["category_id"])
        assert hit.sum() == 1, f"고칠 박스를 못 찾음: {f['file_name']} {f['category_id']}"
        ann.loc[hit, "bbox"] = pd.Series([json.loads(f["bbox_after"])], index=ann.index[hit])
    applied[f["action"]] += 1

print("적용:", applied)
print("박스:", len(ann), "/ 이미지:", ann["file_name"].nunique())

변환 확인: YOLO txt를 다시 읽어서 사진에 그려봄. 박스가 알약에 맞으면 변환 공식이 맞는 것.

In [ ]:
import random
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

random.seed(0)
samples = random.sample(sorted((YOLO_DIR / "images" / "train").glob("*.png")), 4)

fig, axes = plt.subplots(1, 4, figsize=(16, 6))
for ax, img_path in zip(axes, samples):
    im = Image.open(img_path)
    W, H = im.size
    ax.imshow(im)

    label_path = YOLO_DIR / "labels" / "train" / (img_path.stem + ".txt")
    for line in label_path.read_text().splitlines():
        c, cx, cy, bw, bh = line.split()
        # 0~1 비율 → 픽셀, 중심 → 왼쪽 위
        bw, bh = float(bw) * W, float(bh) * H
        x1, y1 = float(cx) * W - bw / 2, float(cy) * H - bh / 2
        ax.add_patch(patches.Rectangle((x1, y1), bw, bh, fill=False, edgecolor="lime", linewidth=2))
        ax.text(x1, y1 - 10, YOLO2CAT[int(c)], color="lime", fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.show()

## 4. 학습

CPU라서 제일 작은 모델(yolo26n)로 3에폭만. 파이프라인이 이어지는지 확인용.
GPU 환경에선 모델 크기와 에폭을 늘릴 것.
- yolo26대신 yolo26n으로 해도 되는 이유: 설계는 다 같고 층의 개수와 폭만 다름. 같은 차종인데 배기량만 다른 차라고 보면 됨.

오늘은 CPU에서 **"끝까지 이어지는지 확인"** 하는 게 목적이라 제일 가벼운 n이 맞음. 대신 **Faster R-CNN과 제대로 비교할 때는** GPU 환경에서 s나 m으로 키워야 함. n은 너무 작아서 비교가 불공평해질 수 있기때문.

코드에서 `MODEL = "yolo26n"` 한 줄만 바꾸면 크기를 바꿀 수 있게 해둠.

In [ ]:
from ultralytics import YOLO, settings

settings.update({"wandb": False})  # W&B는 다음 단계에서 연결

MODEL = "yolo26n"
EPOCHS = 3
SEED = 0
RUN_NAME = f"{MODEL}_e{EPOCHS}_s{SEED}"  # 실행 이름에 모델, 에폭, 시드를 넣어서 구분

model = YOLO(f"{MODEL}.pt")
results = model.train(
    data=str(YOLO_DIR / "data.yaml"),
    epochs=EPOCHS,
    imgsz=640,
    batch=8,
    device=DEVICE,
    seed=SEED,
    workers=0,                    # 윈도우 노트북 에러 방지
    project=str(ROOT / "runs"),
    name=RUN_NAME,
    exist_ok=True,
)

3에폭이라 점수는 의미 없음. 끝까지 도는지 확인용.
대회 지표가 mAP 0.75~0.95라서 그 구간 점수도 따로 계산함.

In [ ]:
m = results.results_dict
print(f"mAP50     : {m['metrics/mAP50(B)']:.3f}")
print(f"mAP50-95  : {m['metrics/mAP50-95(B)']:.3f}")

# 대회 지표: IoU 0.75~0.95 구간 평균
# all_ap는 (클래스 수, IoU 10단계) 표. 10단계 = 0.50, 0.55, ..., 0.95 → 6번째(0.75)부터 끝까지
ap = results.box.all_ap
print(f"mAP75-95  : {ap[:, 5:].mean():.3f}  ← 대회 지표 근사 (val 기준)")

In [ ]:
# 학습 결과 폴더에서 검증 라벨 이미지 보기
from IPython.display import Image, display

run_dir = ROOT / "runs" / RUN_NAME
print(run_dir)

display(Image(filename=run_dir / "val_batch0_labels.jpg"))

## 5. test 예측 → 제출 csv

3에폭 모델로 제출 형식부터 맞춰봄. 점수는 신경 안 씀.
- category_id: 모델 번호(0~55) → 원래 약 번호 (data.yaml 이름에 들어 있음)
- 박스: x1,y1,x2,y2 → x,y,w,h
- mAP 0.75~0.95라 conf는 낮게 둠

In [ ]:
import re
import torch
import pandas as pd
from ultralytics import YOLO

DEVICE = 0 if torch.cuda.is_available() else "cpu"  # 내 컴퓨터는 cpu

PRED_RUN = "yolo26s_e100_s0"           # 예측에 쓸 모델 폴더
run_dir = ROOT / "runs" / PRED_RUN

TEST_IMG_DIR = DATA_DIR / "test_images"
test_paths = sorted(TEST_IMG_DIR.glob("*.png"))
print("test 이미지:", len(test_paths), "/ 예시:", [p.name for p in test_paths[:3]])

model = YOLO(run_dir / "weights" / "best.pt")

rows = []
for r in model.predict(
    source=str(TEST_IMG_DIR),   # 폴더째 넘겨야 한 장씩 읽음
    imgsz=640,
    conf=0.01,      # 낮게 둬야 mAP에 유리
    device=DEVICE,
    stream=True,    # 한 장씩 처리해서 메모리 아끼기
    verbose=False,
):
    image_id = int(re.findall(r"\d+", Path(r.path).stem)[-1])  # 파일 이름에서 숫자만

    for (x1, y1, x2, y2), c, s in zip(r.boxes.xyxy.tolist(), r.boxes.cls.tolist(), r.boxes.conf.tolist()):
        rows.append({
            "image_id": image_id,
            "category_id": int(model.names[int(c)]),  # 0~55 → 원래 약 번호
            "bbox_x": round(x1, 2),
            "bbox_y": round(y1, 2),
            "bbox_w": round(x2 - x1, 2),
            "bbox_h": round(y2 - y1, 2),
            "score": round(s, 4),
        })

sub = pd.DataFrame(rows)
sub.insert(0, "annotation_id", range(1, len(sub) + 1))  # 1부터 순서대로

SUB_PATH = run_dir / "submission.csv"
sub.to_csv(SUB_PATH, index=False)

print("저장:", SUB_PATH)
print("행 수:", len(sub), "/ 예측 나온 이미지:", sub["image_id"].nunique(), "/", len(test_paths))
sub.head()